# NOORIX-AI Printed OCR Training
**Prepared by Muhammad Farhan | NOORIX-AI**
\nThis notebook fine-tunes a pretrained TrOCR model on your own labelled printed image dataset. It evaluates Character Error Rate (CER), Word Error Rate (WER), and saves the trained model. It does not magically create labels: each training image must have its exact transcription.
\n**Language note:** The selected pretrained checkpoint is English-oriented. Urdu can be included only as an experimental fine-tuning task when your CSV contains accurate Urdu transcriptions and enough representative labelled data. Do not treat an English-only checkpoint as validated Urdu OCR.
\n## Dataset CSV format
Create a CSV with two columns: `image_path` and `text`. Each row is one image containing a line (or short text crop) and its exact transcription. Example:
\n| image_path | text |
|---|---|
| lines/img_0001.png | This is a printed line. |
| lines/img_0002.png | Exact text from the second line. |
\nPaths may be absolute Kaggle paths or relative to the dataset folder. Split train/validation automatically if no split column is provided.
\n### Use printed book/document line crops. For Urdu, use accurately transcribed Urdu rows; the English pretrained model is only a starting point and must be evaluated carefully.


In [ ]:
# Kaggle OCR training notebook
import os, glob, json, random, zipfile, shutil, textwrap
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
import torch
print("Python:", __import__("sys").version)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
INPUT_ROOTS = [Path(p) for p in glob.glob("/kaggle/input/*")]
print("Kaggle inputs:")
for p in INPUT_ROOTS: print(" -", p)



In [ ]:
# Find annotation CSV files. A usable CSV needs one image-path column and one transcription column.
CSV_CANDIDATES = []
for root in INPUT_ROOTS:
    CSV_CANDIDATES += list(root.rglob("*.csv"))
print("CSV files found:", len(CSV_CANDIDATES))
for i, p in enumerate(CSV_CANDIDATES[:80]):
    print(f"[{i}] {p}")



In [ ]:
# Select your annotation CSV after reviewing the list above.
# Required columns: image_path (or file_name/path/image) and text (or transcription/label).
# If your dataset has no transcription CSV, create one and upload it as a Kaggle Dataset.
ANNOTATION_CSV = ""  # e.g. "/kaggle/input/my-ocr-data/metadata.csv"
IMAGE_COL = ""       # leave blank for auto-detection
TEXT_COL = ""        # leave blank for auto-detection

if not ANNOTATION_CSV:
    raise ValueError("Set ANNOTATION_CSV to the CSV containing image paths and exact transcriptions. Folder names alone are not OCR ground-truth text.")
df = pd.read_csv(ANNOTATION_CSV)
print("Columns:", list(df.columns))



In [ ]:
# Normalize annotation columns and resolve image paths.
from pathlib import Path
import os
cols_lower = {c.lower().strip(): c for c in df.columns}
def choose_col(explicit, options):
    if explicit:
        if explicit not in df.columns: raise ValueError(f"Column {explicit!r} not found")
        return explicit
    for name in options:
        if name in cols_lower: return cols_lower[name]
    raise ValueError(f"Could not find any of these columns: {options}. Set IMAGE_COL/TEXT_COL manually.")
img_col = choose_col(IMAGE_COL, ["image_path","filepath","file_path","file_name","filename","path","image"])
txt_col = choose_col(TEXT_COL, ["text","transcription","transcript","label","ground_truth","gt"])
def resolve_path(value):
    p = Path(str(value))
    if p.is_file(): return str(p)
    for root in INPUT_ROOTS:
        q = root / p
        if q.is_file(): return str(q)
        matches = list(root.rglob(p.name))
        if matches: return str(matches[0])
    return None
df = df[[img_col, txt_col]].copy()
df.columns = ["image_path", "text"]
df["image_path"] = df["image_path"].apply(resolve_path)
df["text"] = df["text"].fillna("").astype(str).str.strip()
df = df.dropna(subset=["image_path"])
df = df[df["text"].str.len() > 0].reset_index(drop=True)
print("Usable labelled rows:", len(df))
display(df.head())
if len(df) < 20: raise ValueError("Too few labelled rows. Add a larger correctly transcribed dataset.")



In [ ]:
# Train/validation split and basic image integrity checks.
from sklearn.model_selection import train_test_split
random.seed(42)
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
print("Train:", len(train_df), "Validation:", len(val_df))
def check_image(path):
    try:
        with Image.open(path) as im: im.verify()
        return True
    except Exception: return False
for part in (train_df, val_df):
    part["valid"] = part["image_path"].apply(check_image)
    part.drop(part[~part["valid"]].index, inplace=True)
print("After image checks:", len(train_df), len(val_df))



In [ ]:
!pip -q install -U transformers accelerate jiwer sentencepiece evaluate



In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
from torch.utils.data import Dataset
import evaluate
MODEL_NAME = "microsoft/trocr-base-printed"
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.max_length = 128
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
if torch.cuda.is_available():
    model.to("cuda")
print("Loaded:", MODEL_NAME)



In [ ]:
class OCRDataset(Dataset):
    def __init__(self, frame, processor, max_target_length=128):
        self.frame = frame.reset_index(drop=True)
        self.processor = processor
        self.max_target_length = max_target_length
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row.image_path).convert("RGB")
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze(0)
        labels = self.processor.tokenizer(
            row.text, padding="max_length", max_length=self.max_target_length,
            truncation=True, return_tensors="pt"
        ).input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}
train_ds = OCRDataset(train_df, processor)
val_ds = OCRDataset(val_df, processor)
def collate(batch):
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels": torch.stack([x["labels"] for x in batch])
    }



In [ ]:
# Fine-tuning. Start with 3 epochs; increase only after checking validation CER.
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = pred.predictions
    if isinstance(pred_ids, tuple): pred_ids = pred_ids[0]
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_text = processor.batch_decode(label_ids, skip_special_tokens=True)
    return {
        "cer": cer_metric.compute(predictions=pred_text, references=label_text),
        "wer": wer_metric.compute(predictions=pred_text, references=label_text)
    }
args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/noorix_ocr_checkpoints",
    predict_with_generate=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    report_to="none"
)
trainer = Seq2SeqTrainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collate, compute_metrics=compute_metrics
)
trainer.train()



In [ ]:
# Final validation metrics and a few sample predictions.
metrics = trainer.evaluate()
print(metrics)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
def predict(path):
    image = Image.open(path).convert("RGB")
    px = processor(images=image, return_tensors="pt").pixel_values.to(device)
    with torch.no_grad():
        ids = model.generate(px, max_new_tokens=128)
    return processor.batch_decode(ids, skip_special_tokens=True)[0]
for _, row in val_df.head(8).iterrows():
    print("\nREFERENCE:", row.text)
    print("PREDICTED:", predict(row.image_path))



In [ ]:
# Save model, processor, metadata and one ZIP for download.
OUT = Path("/kaggle/working/NOORIX_PRINTED_OCR_Muhammad_Farhan")
OUT.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUT / "model")
processor.save_pretrained(OUT / "processor")
meta = {
    "project": "NOORIX-AI", "prepared_by": "Muhammad Farhan",
    "task": "printed OCR", "base_model": MODEL_NAME,
    "train_samples": len(train_df), "validation_samples": len(val_df),
    "validation_metrics": {k: float(v) for k,v in metrics.items() if isinstance(v, (int,float))},
    "languages": "Checkpoint is English-oriented; Urdu fine-tuning is experimental and requires verified Urdu labels.",
    "warning": "Evaluate on a separate real-world test set before integration."
}
(OUT / "metadata.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
zip_path = shutil.make_archive(str(OUT), "zip", OUT)
print("Saved:", zip_path)



## Before integrating into NOORIX-AI
Check validation CER/WER and visually inspect predictions. Lower CER/WER is better. Keep the original held-out test set untouched. Do not claim Urdu support or production accuracy until tested on a representative Urdu test set. The saved Hugging Face model folder is the primary artifact; ONNX export should be handled in a separate compatibility-tested step.
